# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API
## Customer Churn Prediction

**Objective:**
Build a reusable, production-ready machine learning pipeline that predicts
whether a telecom customer will churn (cancel their subscription), using the
scikit-learn `Pipeline` API to combine preprocessing and modeling into a
single deployable object.

**Tools and Technologies:**
- Python 3
- pandas, numpy — data loading and cleaning
- scikit-learn — `Pipeline`, `ColumnTransformer`, `StandardScaler`,
  `OneHotEncoder`, `SimpleImputer`, `LogisticRegression`,
  `RandomForestClassifier`, `GridSearchCV`, evaluation metrics
- joblib — exporting the trained pipeline for reuse/deployment

**Dataset:**
Telco Customer Churn Dataset (IBM Sample Data Set)
Kaggle source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

7,043 customer records, 21 columns (demographics, account info, subscribed
services, and the target column `Churn`).

**Skills Gained:**
- ML pipeline construction
- Hyperparameter tuning with `GridSearchCV`
- Model export and reusability
- Production-readiness practices

**Important note before running:** place the dataset CSV file in the same
folder as this notebook. The loader cell below looks for the two most common
filenames (`telco.csv` and `WA_Fn-UseC_-Telco-Customer-Churn.csv`) and falls
back to downloading a mirrored copy from GitHub if neither is found locally.


## Step 1: Import Libraries

**Purpose:** Load every library used across the notebook — data handling
(pandas/numpy), pipeline building blocks, models, hyperparameter search, the
evaluation metrics we'll use to compare models, and `joblib` for exporting
the final pipeline.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


## Step 2: Load the Dataset

**Purpose:** Read the Telco Customer Churn CSV into a pandas DataFrame and
take a first look at its shape and structure.

**Dataset link:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [2]:
candidate_files = ["telco.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv", "Telco-Customer-Churn.csv"]
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

local_path = next((f for f in candidate_files if os.path.exists(f)), None)

if local_path:
    df = pd.read_csv(local_path)
    print(f"Loaded local file: {local_path}")
else:
    df = pd.read_csv(DATA_URL)
    print("Local file not found — loaded dataset from GitHub mirror instead.")

print(f"Shape: {df.shape}")
df.head()

Local file not found — loaded dataset from GitHub mirror instead.
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 3: Quick Data Inspection

**Purpose:** Check column data types, missing values, and the target class
balance before doing any cleaning. This tells us what preprocessing steps
will actually be needed.

**Important info:** `TotalCharges` looks numeric but is stored as text
because a small number of new customers (0 months tenure) have a blank
value instead of a number — this needs to be fixed before modeling.

In [3]:
df.info()
print("\nMissing values per column:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nChurn class balance:\n", df["Churn"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Step 4: Data Cleaning

**Purpose:**
- Drop `customerID` — it's just a unique identifier, not a predictive
  feature.
- Convert `TotalCharges` from text to numeric (blank strings become `NaN`,
  which we'll handle safely inside the pipeline later with an imputer — this
  avoids data leakage since the imputer will only ever be fit on training
  data).
- Encode the target `Churn` column as 1 (Yes) / 0 (No) so it works with
  scikit-learn classifiers and metrics.

In [4]:
df = df.drop(columns=["customerID"])

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (7043, 19)
Target shape: (7043,)


## Step 5: Train / Test Split

**Purpose:** Hold out 20% of the data as an untouched test set to fairly
evaluate the final models. `stratify=y` keeps the churn/no-churn ratio the
same in both splits, which matters here since the classes are imbalanced
(~73% no-churn vs ~27% churn).

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")

Train set: (5634, 19)
Test set:  (1409, 19)


## Step 6: Identify Numeric vs Categorical Columns

**Purpose:** Numeric and categorical features need different preprocessing
(scaling vs encoding), so we separate them here to feed into the
`ColumnTransformer` in the next step.

In [6]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numeric_features]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

Numeric features (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features (16): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Step 7: Build the Preprocessing Pipeline (ColumnTransformer)

**Purpose:** This is the core reusability piece. We build two small
pipelines:
- **Numeric pipeline:** impute missing values with the median, then scale
  with `StandardScaler` (important for Logistic Regression).
- **Categorical pipeline:** impute missing values with the most frequent
  category, then apply `OneHotEncoder` (`handle_unknown="ignore"` so the
  pipeline won't break if it ever sees a new category in production data).

`ColumnTransformer` then routes each column type to the right pipeline and
concatenates the results into one feature matrix.

In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

print("Preprocessor built successfully.")

Preprocessor built successfully.


## Step 8: Build Full Model Pipelines

**Purpose:** Chain the preprocessor into two full pipelines — one ending in
`LogisticRegression`, one ending in `RandomForestClassifier`. Because
preprocessing lives inside the pipeline, calling `.fit()` on either of these
runs cleaning + encoding + scaling + training in one call, and the exact
same steps are automatically applied whenever we call `.predict()` later —
including on brand new, raw customer data.

In [8]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE)),
])

print("Pipelines built:")
print(" - Logistic Regression pipeline")
print(" - Random Forest pipeline")

Pipelines built:
 - Logistic Regression pipeline
 - Random Forest pipeline


## Step 9: Define Hyperparameter Grids

**Purpose:** Specify the hyperparameter values `GridSearchCV` will search
over for each model. Note the `classifier__<param>` naming — this tells
scikit-learn to set that parameter on the step named `"classifier"` inside
the pipeline.

- **Logistic Regression:** `C` (inverse regularization strength), `penalty`,
  and `solver`.
- **Random Forest:** `n_estimators` (number of trees), `max_depth`, and
  `min_samples_split`.

In [9]:
logreg_param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs", "liblinear"],
}

rf_param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__min_samples_split": [2, 5, 10],
}

models = {
    "LogisticRegression": (logreg_pipeline, logreg_param_grid),
    "RandomForest": (rf_pipeline, rf_param_grid),
}

print("Hyperparameter grids defined.")

Hyperparameter grids defined.


## Step 10: Hyperparameter Tuning with GridSearchCV

**Purpose:** For each model, run 5-fold cross-validated grid search scored
on ROC-AUC (a good metric here since the classes are imbalanced). This finds
the best hyperparameter combination for each model type and evaluates the
resulting best pipeline on the held-out test set.

**Important info:** `n_jobs=-1` uses all available CPU cores to speed up the
search. This cell may take a couple of minutes to run, mainly due to the
Random Forest grid.

In [10]:
results = {}
best_estimators = {}

for name, (pipeline, param_grid) in models.items():
    sep = "=" * 60
    print(f"\n{sep}\nTuning {name}\n{sep}")
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=1,
    )
    grid_search.fit(X_train, y_train)

    best_estimators[name] = grid_search.best_estimator_
    print(f"Best params: {grid_search.best_params_}")
    print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

    y_pred = grid_search.predict(X_test)
    y_proba = grid_search.predict_proba(X_test)[:, 1]

    results[name] = {
        "best_params": grid_search.best_params_,
        "cv_roc_auc": grid_search.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_proba),
    }

    print(f"\nTest set performance for {name}:")
    for k, v in results[name].items():
        if k != "best_params":
            print(f"  {k}: {v:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))


Tuning LogisticRegression
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'classifier__C': 10, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV ROC-AUC: 0.8459

Test set performance for LogisticRegression:
  cv_roc_auc: 0.8459
  test_accuracy: 0.8048
  test_precision: 0.6552
  test_recall: 0.5588
  test_f1: 0.6032
  test_roc_auc: 0.8411

Classification report:
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.66      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409


Tuning RandomForest
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'classifier__max_depth': 10, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}
Best CV ROC-AUC: 0.8454

Test set performance for RandomForest:


## Step 11: Compare Models and Select the Best One

**Purpose:** Lay both models' results side by side and pick the one with the
highest test-set ROC-AUC as the final production model.

In [11]:
summary_df = pd.DataFrame(results).T
display(summary_df)

best_model_name = max(results, key=lambda k: results[k]["test_roc_auc"])
best_pipeline = best_estimators[best_model_name]

print(f"\nBest overall model: {best_model_name}")
print(f"Test ROC-AUC: {results[best_model_name]['test_roc_auc']:.4f}")

,best_params,cv_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
LogisticRegression,"{'classifier__C': 10, 'classifier__penalty': '...",0.845882,0.804826,0.655172,0.558824,0.603175,0.841127
RandomForest,"{'classifier__max_depth': 10, 'classifier__min...",0.845396,0.803407,0.665529,0.52139,0.584708,0.840759



Best overall model: LogisticRegression
Test ROC-AUC: 0.8411


## Step 12: Confusion Matrix for the Best Model

**Purpose:** Look beyond the summary metrics to see exactly how many
customers were correctly/incorrectly classified as churn vs no-churn.

In [12]:
y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Churn", "Actual: Churn"],
    columns=["Predicted: No Churn", "Predicted: Churn"],
)
display(cm_df)

,Predicted: No Churn,Predicted: Churn
Actual: No Churn,925,110
Actual: Churn,165,209


## Step 13: Export the Complete Pipeline with joblib

**Purpose:** Save the entire fitted pipeline — preprocessing steps AND the
tuned model — as a single `.joblib` file. This is the key production-
readiness step: anyone can load this one file and call `.predict()` directly
on raw, unprocessed customer records, with no separate preprocessing code
needed.

In [13]:
output_path = "churn_pipeline.joblib"
joblib.dump(best_pipeline, output_path)
print(f"Saved complete pipeline to: {output_path}")

Saved complete pipeline to: churn_pipeline.joblib


## Step 14: Sanity Check — Reload and Predict

**Purpose:** Prove the exported file is truly self-contained by loading it
back from disk (simulating a fresh deployment) and running predictions on a
few raw test rows straight away.

In [14]:
loaded_pipeline = joblib.load(output_path)

sample = X_test.iloc[:5]
sample_preds = loaded_pipeline.predict(sample)
sample_probs = loaded_pipeline.predict_proba(sample)[:, 1]

for i, (pred, prob) in enumerate(zip(sample_preds, sample_probs)):
    label = "Churn" if pred == 1 else "No Churn"
    print(f"Row {i}: predicted={label}  churn_probability={prob:.3f}")

Row 0: predicted=No Churn  churn_probability=0.047
Row 1: predicted=Churn  churn_probability=0.683
Row 2: predicted=No Churn  churn_probability=0.050
Row 3: predicted=No Churn  churn_probability=0.423
Row 4: predicted=No Churn  churn_probability=0.023


## Conclusion & Results

**What was built:**
A complete, reusable scikit-learn `Pipeline` that takes raw Telco customer
records and outputs a churn prediction, with all preprocessing (imputation,
scaling, one-hot encoding) and modeling bundled into a single deployable
object. Two candidate models — Logistic Regression and Random Forest — were
each tuned with 5-fold `GridSearchCV`, and the best-performing model
(by test ROC-AUC) was selected and exported with `joblib`.

**Results (fill in with the numbers your run produces above):**
- Logistic Regression — best CV ROC-AUC: *see `results["LogisticRegression"]`*
- Random Forest — best CV ROC-AUC: *see `results["RandomForest"]`*
- Final selected model: *see `best_model_name`* saved to `churn_pipeline.joblib`

**Why this is production-ready:**
- No manual preprocessing step is ever separated from the model — they are
  fit and shipped together, eliminating train/serve skew.
- `OneHotEncoder(handle_unknown="ignore")` means the pipeline won't crash on
  unseen categories in future data.
- The whole pipeline is a single artifact (`churn_pipeline.joblib`) that can
  be loaded in any Python service and called with `.predict()` /
  `.predict_proba()` directly on raw input rows.

**Possible next steps:**
- Try additional models (Gradient Boosting, XGBoost/LightGBM).
- Address class imbalance further with class weighting or SMOTE.
- Add SHAP/feature-importance analysis to explain predictions to the
  business (e.g. which factors drive churn most).
- Wrap the loaded pipeline in a simple API (Flask/FastAPI) for real-time
  scoring.
